# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: /Users/parthdagia/flasheats-classroom-pack/divvyRebalancingWithMyIQ/flasheats-challenges


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
LATE_THRESHOLD_MIN = 10   # our KPI definition from Classes 5 and 6 (VP Ops to confirm)

base_orders = orders.drop_duplicates("order_id", keep="first").copy()

# order_outcomes.late_flag uses ANY delay > 0 (843 "late" = the 56% rule).
# We keep it for comparison but select examples with our > 10 min definition.
outcomes["late_10"] = outcomes["delay_min"] > LATE_THRESHOLD_MIN
print("late_flag (any delay):", int(outcomes["late_flag"].sum()),
      "| late_10 (> 10 min):", int(outcomes["late_10"].sum()))

ts = lambda s: pd.to_datetime(s, format="mixed", errors="coerce")
driver_events = pd.DataFrame([{"driver_id": d["driver_id"], **e}
                              for d in json.load(open(BASE / "data" / "driver_events.json")) for e in d["events"]])
dispatch = pd.DataFrame([r for p in sorted((BASE / "student_output" / "raw_dispatch").glob("page_*.json"))
                         for r in json.loads(p.read_text())["data"]])     # raw pages preserved in Class 5
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv").drop_duplicates()

def build_order_timeline(order_id):
    o = base_orders.loc[base_orders["order_id"] == order_id].iloc[0]
    rows = [
        (o["created_at"], "ORDER_CREATED", f"customer {o['customer_id']}", "orders (SQLite)"),
        (o["promised_eta"], "ETA_PROMISED (target)", "system", "orders (SQLite)"),
        (o["pickup_at"], "PICKED_UP", f"driver {o['driver_id']}", "orders (SQLite)"),
        (o["actual_delivery_at"], "DELIVERED", f"driver {o['driver_id']}", "orders (SQLite)"),
    ]
    d = dispatch.loc[dispatch["order_id"] == order_id]
    for _, r in d.iterrows():
        rows.append((r["assigned_at"], "DRIVER_ASSIGNED", f"driver {r['original_driver_id']}", "dispatch API"))
        rows.append((r["estimated_pickup_at"], "PICKUP_ESTIMATED (target)", "dispatch", "dispatch API"))
        if pd.notna(r["reassigned_at"]):
            rows.append((r["reassigned_at"], "DRIVER_REASSIGNED", f"driver {r['driver_id']}", "dispatch API"))
    for _, r in restaurant_status.loc[restaurant_status["order_id"] == order_id].iterrows():
        rows.append((r["last_updated_at"], f"RESTAURANT_STATUS: {r['status'].strip().lower()}", f"restaurant {r['restaurant_id']}", "restaurant_status.csv"))
    for _, r in customer_actions.loc[customer_actions["order_id"] == order_id].iterrows():
        rows.append((r["action_at"], r["action_type"], f"customer {r['customer_id']}", "customer_app_actions.csv"))
    for _, r in tickets.loc[tickets["order_id"] == order_id].drop_duplicates("ticket_id").iterrows():
        rows.append((r["created_at"], f"SUPPORT_TICKET: {r['category']}", "customer", "support_tickets.csv"))
    for _, r in interventions.loc[interventions["order_id"] == order_id].iterrows():
        rows.append((r["intervention_at"], f"INTERVENTION: {r['intervention_type']} ({r['reason']})", r["initiated_by"], "order_interventions.csv"))
    pings = driver_events[(driver_events["order_id"] == order_id) & (driver_events["type"] == "gps_ping")]
    for _, r in pings.iterrows():
        rows.append((r["timestamp"], "GPS_PING", f"driver {r['driver_id']}", "driver_events.json"))
    tl = pd.DataFrame(rows, columns=["event_time", "event_type", "actor", "source_system"])
    tl["event_time"] = ts(tl["event_time"])
    tl = tl.dropna(subset=["event_time"]).sort_values("event_time").reset_index(drop=True)
    tl["min_since_order"] = ((tl["event_time"] - tl["event_time"].min()).dt.total_seconds() / 60).round(1)
    return tl

on_time_order = outcomes.loc[outcomes["outcome_bucket"].eq("delivered_on_time"), "order_id"].iloc[0]
late_order = outcomes.loc[outcomes["late_10"] & ~outcomes["order_id"].isin(interventions["order_id"]), "order_id"].iloc[0]
# an intervention order that is also late, to see whether the intervention came in time
iv_late = interventions.merge(outcomes, on="order_id")
intervention_order = iv_late.loc[iv_late["late_10"] & (iv_late["intervention_type"] != "CUSTOMER_CREDIT"), "order_id"].iloc[0]
print("on-time", on_time_order, "| late", late_order, "| intervention", intervention_order)

for label, oid in [("ON TIME", on_time_order), ("LATE", late_order), ("INTERVENTION + LATE", intervention_order)]:
    delay = outcomes.set_index("order_id").loc[oid, "delay_min"]
    print(f"\n=== {label}: {oid} (delay vs ETA: {delay:+.1f} min)")
    display(build_order_timeline(oid))

# Cross-source check found while building timelines: does dispatch confirm reassignment interventions?
reassign_iv = interventions.loc[interventions["intervention_type"] == "DRIVER_REASSIGNMENT", "order_id"]
dispatch_reassigned = dispatch.loc[dispatch["reassigned_at"].notna(), "order_id"]
print(f"DRIVER_REASSIGNMENT interventions: {len(reassign_iv)} | dispatch reassigned_at set: {len(dispatch_reassigned)} "
      f"| in both: {reassign_iv.isin(dispatch_reassigned).sum()}")

late_flag (any delay): 843 | late_10 (> 10 min): 349
on-time O00003 | late O00005 | intervention O00781

=== ON TIME: O00003 (delay vs ETA: -4.4 min)


,event_time,event_type,actor,source_system,min_since_order
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer C0855,orders (SQLite),0.0
1,2026-08-01 19:35:00.002443,DRIVER_ASSIGNED,driver D039,dispatch API,0.0
2,2026-08-01 19:44:37.017130,GPS_PING,driver D039,driver_events.json,9.6
3,2026-08-01 19:48:00.000000,RESTAURANT_STATUS: preparing,restaurant R010,restaurant_status.csv,13.0
4,2026-08-01 19:53:00.000000,PICKUP_ESTIMATED (target),dispatch,dispatch API,18.0
5,2026-08-01 19:54:14.031817,GPS_PING,driver D039,driver_events.json,19.2
6,2026-08-01 19:57:50.820366,PICKED_UP,driver D039,orders (SQLite),22.8
7,2026-08-01 20:03:51.046504,GPS_PING,driver D039,driver_events.json,28.9
8,2026-08-01 20:13:28.061191,DELIVERED,driver D039,orders (SQLite),38.5
9,2026-08-01 20:17:52.941967,ETA_PROMISED (target),system,orders (SQLite),42.9



=== LATE: O00005 (delay vs ETA: +21.0 min)


,event_time,event_type,actor,source_system,min_since_order
0,2026-08-06 20:35:00.000000,ORDER_CREATED,customer C0040,orders (SQLite),0.0
1,2026-08-06 20:36:55.346822,DRIVER_ASSIGNED,driver D047,dispatch API,1.9
2,2026-08-06 20:53:00.000000,PICKUP_ESTIMATED (target),dispatch,dispatch API,18.0
3,2026-08-06 21:09:53.328119,GPS_PING,driver D047,driver_events.json,34.9
4,2026-08-06 21:18:38.024280,PICKED_UP,driver D047,orders (SQLite),43.6
5,2026-08-06 21:42:51.309416,GPS_PING,driver D047,driver_events.json,67.9
6,2026-08-06 21:54:48.000000,ETA_PROMISED (target),system,orders (SQLite),79.8
7,2026-08-06 22:15:49.290713,DELIVERED,driver D047,orders (SQLite),100.8



=== INTERVENTION + LATE: O00781 (delay vs ETA: +13.1 min)


,event_time,event_type,actor,source_system,min_since_order
0,2026-08-22 12:33:00.000000,ORDER_CREATED,customer C0103,orders (SQLite),0.0
1,2026-08-22 12:40:42.931820,DRIVER_ASSIGNED,driver D103,dispatch API,7.7
2,2026-08-22 12:51:00.000000,PICKUP_ESTIMATED (target),dispatch,dispatch API,18.0
3,2026-08-22 12:54:00.000000,ETA_VIEWED,customer C0103,customer_app_actions.csv,21.0
4,2026-08-22 12:57:00.000000,INTERVENTION: PRIORITY_DISPATCH (late_risk),operations,order_interventions.csv,24.0
5,2026-08-22 13:04:00.000000,RESTAURANT_STATUS: ready,restaurant R019,restaurant_status.csv,31.0
6,2026-08-22 13:07:42.051142,GPS_PING,driver D103,driver_events.json,34.7
7,2026-08-22 13:11:17.309762,PICKED_UP,driver D103,orders (SQLite),38.3
8,2026-08-22 13:22:41.170463,GPS_PING,driver D103,driver_events.json,49.7
9,2026-08-22 13:48:36.000000,ETA_PROMISED (target),system,orders (SQLite),75.6


DRIVER_REASSIGNMENT interventions: 155 | dispatch reassigned_at set: 95 | in both: 8


### My answer

| Order | Outcome | What I saw |
|:-|:-|:-|
| O00003 | 4 min early | Picked up 5 min after the estimate, short ride |
| O00005 | 21 min late | Picked up **26 min after** the estimate. The time was lost before pickup |
| O00781 | 13 min late | Priority dispatch at 24 min, still late |

Things I noticed:
- Dispatch's estimated pickup is always exactly 18 min after ordering.
- Intervention log and dispatch don't agree: only **8 of 155** reassignments show up in dispatch.
- `order_outcomes.late_flag` uses "any delay" (843 late = 56%), so I used my own `late_10` (> 10 min).

# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [4]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))

# Canonical model: one table per business concept, keyed around the ORDER
t_clean = tickets.drop_duplicates("ticket_id")
customer_interaction = pd.concat([
    customer_actions.rename(columns={"action_id": "interaction_id", "action_type": "interaction_type",
                                     "action_at": "interaction_at"})[["interaction_id", "order_id", "customer_id", "interaction_type", "interaction_at", "channel"]],
    t_clean.dropna(subset=["order_id"]).merge(base_orders[["order_id", "customer_id"]], on="order_id")
           .assign(interaction_type="SUPPORT_TICKET", channel="support")
           .rename(columns={"ticket_id": "interaction_id", "created_at": "interaction_at"})
           [["interaction_id", "order_id", "customer_id", "interaction_type", "interaction_at", "channel"]],
], ignore_index=True)

model = {
    "customer":             (customers, ["customer_id"], [], "one customer"),
    "order":                (base_orders, ["order_id"], ["customer_id", "restaurant_id", "driver_id"], "one order (deduplicated)"),
    "customer_interaction": (customer_interaction, ["interaction_id"], ["order_id", "customer_id"], "one customer touchpoint (app action or ticket)"),
    "intervention":         (interventions, ["intervention_id"], ["order_id"], "one action FlashEats took on an order"),
    "order_outcome":        (outcomes, ["order_id"], ["order_id"], "one final result per order"),
}
parents = {"customer_id": customers["customer_id"], "order_id": base_orders["order_id"],
           "restaurant_id": restaurants["restaurant_id"], "driver_id": drivers["driver_id"]}
doc = []
for table, (df, pk, fks, grain) in model.items():
    fk_ok = {fk: f"{df[fk].dropna().isin(parents[fk]).mean():.1%}" for fk in fks}
    doc.append({"table": table, "grain": grain, "rows": len(df), "primary_key": ", ".join(pk),
                "pk_unique": not df.duplicated(pk).any(),
                "foreign_keys (resolve %)": ", ".join(f"{k} ({v})" for k, v in fk_ok.items()) or "none"})
display(pd.DataFrame(doc))

print("Orders per customer (median):", base_orders.groupby("customer_id").size().median())
print("Interventions per order (max):", interventions.groupby("order_id").size().max())
print("Interactions per order (max):", customer_interaction.groupby("order_id").size().max())

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


...

,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 7)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket,late_10
0,O00001,delivered,1,1.0,10.82,delivered_late,True
1,O00002,delivered,1,1.0,3.97,delivered_late,False


,table,grain,rows,primary_key,pk_unique,foreign_keys (resolve %)
0,customer,one customer,900,customer_id,True,none
1,order,one order (deduplicated),1600,order_id,True,"customer_id (100.0%), restaurant_id (100.0%), driver_id (100.0%)"
2,customer_interaction,one customer touchpoint (app action or ticket),2563,interaction_id,True,"order_id (100.0%), customer_id (100.0%)"
3,intervention,one action FlashEats took on an order,430,intervention_id,True,order_id (100.0%)
4,order_outcome,one final result per order,1600,order_id,True,order_id (100.0%)


Orders per customer (median): 2.0
Interventions per order (max): 1
Interactions per order (max): 5


### My answer

| Table | Grain | PK | FKs |
|:-|:-|:-|:-|
| customer | one customer | customer_id | |
| order | one order (deduplicated) | order_id | customer_id, restaurant_id, driver_id |
| customer_interaction | one app action or ticket | interaction_id | order_id, customer_id |
| intervention | one action by FlashEats | intervention_id | order_id |
| order_outcome | one result per order | order_id | order_id |

All PKs are unique and all FKs match.

**Why this is better than copying every source table:** everything is built around the order, so every question is just "join to order". App "support opened" and support tickets are the same thing (customer reaching out), so they go in one table.

# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [5]:
# 1) Aggregate each one-to-many table to ONE row per order before joining (no fan-out)
actions_by_order = (customer_actions.assign(
        eta_view=customer_actions["action_type"].eq("ETA_VIEWED"),
        support_open=customer_actions["action_type"].eq("SUPPORT_OPENED"),
        cancel_try=customer_actions["action_type"].eq("CANCEL_ATTEMPTED"))
    .groupby("order_id").agg(action_count=("action_id", "count"), eta_views=("eta_view", "sum"),
                             support_app=("support_open", "any"), cancel_attempted=("cancel_try", "any"))
    .reset_index())

tickets_by_order = (tickets.drop_duplicates("ticket_id").dropna(subset=["order_id"])
                    .groupby("order_id").agg(ticket_count=("ticket_id", "count")).reset_index())

# Service recovery (a credit after the fact) is not the same as an operational fix
OPERATIONAL = {"DRIVER_REASSIGNMENT", "PRIORITY_DISPATCH", "RESTAURANT_CONTACT"}
iv_by_order = (interventions.assign(operational=interventions["intervention_type"].isin(OPERATIONAL))
               .groupby("order_id").agg(intervention_count=("intervention_id", "count"),
                                        intervention_types=("intervention_type", lambda s: ", ".join(sorted(set(s)))),
                                        operational_intervention=("operational", "any"))
               .reset_index())

# 2) Join everything to the order grain
order_model = (base_orders[["order_id", "customer_id", "restaurant_id"]]
    .merge(actions_by_order, on="order_id", how="left")
    .merge(tickets_by_order, on="order_id", how="left")
    .merge(iv_by_order, on="order_id", how="left")
    .merge(outcomes[["order_id", "final_status_norm", "late_flag", "late_10", "delay_min", "outcome_bucket"]],
           on="order_id", how="left"))
order_model = order_model.fillna({"action_count": 0, "eta_views": 0, "support_app": False, "cancel_attempted": False,
                                  "ticket_count": 0, "intervention_count": 0, "intervention_types": "",
                                  "operational_intervention": False})
order_model["support_opened"] = order_model["support_app"] | (order_model["ticket_count"] > 0)
order_model = order_model.rename(columns={"final_status_norm": "final_status"})
assert order_model["order_id"].is_unique and len(order_model) == len(base_orders), "fan-out!"

cols = ["order_id", "customer_id", "support_opened", "cancel_attempted", "intervention_count",
        "intervention_types", "final_status", "late_10", "delay_min"]
display(order_model[cols].head(10))

valid = order_model[order_model["delay_min"].notna()]          # delivered with a time
late = valid[valid["late_10"]]
print(f"1. Late orders with a support interaction: {late['support_opened'].sum()} of {len(late)} ({late['support_opened'].mean():.0%})")
print(f"2. Orders that received an intervention:   {(order_model['intervention_count'] > 0).sum()} of {len(order_model)}")
print("3. Most common intervention:")
display(interventions["intervention_type"].value_counts().to_frame("interventions"))
frustrated = valid[(valid["support_opened"] | valid["cancel_attempted"]) & (valid["intervention_count"] == 0)]
print(f"4. Frustrated journeys (support or cancel attempt) with NO intervention: {len(frustrated)}, "
      f"of which late: {frustrated['late_10'].sum()}")
display(frustrated.sort_values("delay_min", ascending=False)[cols].head(10))

,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_10,delay_min
0,O00001,C0168,True,False,1.0,DRIVER_REASSIGNMENT,delivered,True,10.82
1,O00002,C0043,False,False,0.0,,delivered,False,3.97
2,O00003,C0855,False,False,0.0,,delivered,False,-4.41
3,O00004,C0229,False,False,0.0,,cancelled,False,NaN
4,O00005,C0040,False,False,0.0,,delivered,True,21.02
5,O00006,C0152,False,False,0.0,,delivered,False,7.30
6,O00007,C0875,False,False,0.0,,delivered,False,4.27
7,O00008,C0223,True,False,1.0,CUSTOMER_CREDIT,delivered,True,26.86
8,O00009,C0685,False,False,0.0,,delivered,False,0.69
9,O00010,C0530,False,False,0.0,,delivered,False,-5.79


1. Late orders with a support interaction: 186 of 349 (53%)
2. Orders that received an intervention:   430 of 1600
3. Most common intervention:


,interventions
intervention_type,
DRIVER_REASSIGNMENT,155
RESTAURANT_CONTACT,116
PRIORITY_DISPATCH,95
CUSTOMER_CREDIT,64


4. Frustrated journeys (support or cancel attempt) with NO intervention: 218, of which late: 141


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_10,delay_min
193,O00194,C0771,True,False,0.0,,delivered,True,38.34
736,O00737,C0824,True,False,0.0,,delivered,True,36.28
302,O00303,C0411,True,False,0.0,,delivered,True,33.16
210,O00211,C0629,True,False,0.0,,delivered,True,32.38
1234,O01235,C0772,True,False,0.0,,delivered,True,32.00
976,O00977,C0254,True,False,0.0,,delivered,True,31.92
994,O00995,C0746,True,False,0.0,,delivered,True,31.72
1196,O01197,C0032,True,False,0.0,,delivered,True,31.61
892,O00893,C0556,True,False,0.0,,delivered,True,30.32
1143,O01144,C0788,True,False,0.0,,delivered,True,30.13


### My answer

I grouped every table to one row per order first, then joined (1,600 orders in, 1,600 out, no duplicates).
I also split interventions into **operational** (reassign, priority dispatch, restaurant contact) and **customer credit**, because a credit is given after the problem and can't make an order on time.

1. Late orders with support interaction: **186 of 349 (53%)**
2. Orders with an intervention: **430 of 1,600**
3. Most common: **DRIVER_REASSIGNMENT (155)**
4. Frustrated customers (support or cancel attempt) with no intervention: **218**, of which **141 were late**

# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [6]:
# Stage timing from the order and dispatch records
stage = base_orders[["order_id", "created_at", "pickup_at"]].merge(
    dispatch[["order_id", "estimated_pickup_at"]], on="order_id")
for c in ["created_at", "pickup_at", "estimated_pickup_at"]:
    stage[c] = ts(stage[c])
stage["pickup_overrun_min"] = (stage["pickup_at"] - stage["estimated_pickup_at"]).dt.total_seconds() / 60
order_model = order_model.drop(columns=["pickup_overrun_min"], errors="ignore").merge(
    stage[["order_id", "pickup_overrun_min"]], on="order_id", how="left")
valid = order_model[order_model["delay_min"].notna()]

# Intervention timing relative to pickup
ivt = interventions.merge(stage[["order_id", "pickup_at"]], on="order_id")
ivt["before_pickup"] = ts(ivt["intervention_at"]) < ivt["pickup_at"]
op = ivt[ivt["intervention_type"].isin(OPERATIONAL)]

frustrated_all = valid[valid["support_opened"] | valid["cancel_attempted"]]

metrics = pd.DataFrame([
    ("M1 Late Delivery Rate (> 10 min)", "OUTCOME",
     "late_10 orders / delivered orders with a delivery time", "order",
     valid["late_10"].mean(), "The project KPI itself"),
    ("M2 Pickup overrun > 15 min rate", "WORKFLOW DRIVER",
     "orders picked up > 15 min after dispatch's estimated pickup / delivered orders", "order",
     (valid["pickup_overrun_min"] > 15).mean(), "Where delay accumulates; the leading indicator of M1"),
    ("M3 Customer contact rate", "CUSTOMER INTERACTION",
     "delivered orders with support opened or a ticket / delivered orders", "order",
     valid["support_opened"].mean(), "The customer's real-time signal that an order is going wrong"),
    ("M4 Unanswered frustration rate", "INTERVENTION",
     "frustrated journeys with no intervention / frustrated journeys", "order",
     (frustrated_all["intervention_count"] == 0).mean(), "How often a customer flag gets no response"),
    ("M5 Timely operational intervention rate", "INTERVENTION",
     "operational interventions before pickup / operational interventions", "intervention",
     op["before_pickup"].mean(), "An intervention after pickup cannot fix the pre-pickup delay"),
], columns=["metric", "type", "formula", "grain", "value", "why_it_matters"])
metrics["value"] = metrics["value"].map("{:.1%}".format)
display(metrics)

# Is M2 really linked to the KPI? Late rate by pickup overrun
band = pd.cut(valid["pickup_overrun_min"], [-100, 0, 5, 10, 15, 20, 30, 200],
              labels=["early", "0-5", "5-10", "10-15", "15-20", "20-30", "30+"])
display(valid.groupby(band, observed=True)["late_10"].agg(orders="count", late_rate="mean").round(3))

,metric,type,formula,grain,value,why_it_matters
0,M1 Late Delivery Rate (> 10 min),OUTCOME,late_10 orders / delivered orders with a delivery time,order,23.3%,The project KPI itself
1,M2 Pickup overrun > 15 min rate,WORKFLOW DRIVER,orders picked up > 15 min after dispatch's estimated pickup / delivered orders,order,25.8%,Where delay accumulates; the leading indicator of M1
2,M3 Customer contact rate,CUSTOMER INTERACTION,delivered orders with support opened or a ticket / delivered orders,order,19.5%,The customer's real-time signal that an order is going wrong
3,M4 Unanswered frustration rate,INTERVENTION,frustrated journeys with no intervention / frustrated journeys,order,74.4%,How often a customer flag gets no response
4,M5 Timely operational intervention rate,INTERVENTION,operational interventions before pickup / operational interventions,intervention,68.9%,An intervention after pickup cannot fix the pre-pickup delay


,orders,late_rate
pickup_overrun_min,,
early,243,0.000
0-5,298,0.000
5-10,326,0.018
10-15,243,0.136
15-20,151,0.556
20-30,176,0.977
30+,58,0.931


### My answer

| Metric | Type | Formula | Value |
|:-|:-|:-|-:|
| M1 Late rate (> 10 min) | Outcome | late / delivered with time | 23.3% |
| M2 Pickup overrun > 15 min | Workflow | picked up > 15 min after estimate / delivered | 25.8% |
| M3 Customer contact rate | Interaction | orders with support / delivered | 19.5% |
| M4 Unanswered frustration | Intervention | frustrated with no intervention / frustrated | 74.4% |
| M5 Intervention before pickup | Intervention | operational before pickup / all operational | 68.9% |

M2 is the most useful: 0% late when pickup is within 5 min of the estimate, 97% late when it's 20+ min over. So we can see a late order coming and act early.

# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [7]:
def summary(df, by):
    return (df.groupby(by, observed=True)
              .agg(orders=("order_id", "count"), late_rate=("late_10", "mean"), median_delay=("delay_min", "median"))
              .round({"late_rate": 3, "median_delay": 1}))

print("A. Support interaction vs delay")
display(summary(valid, "support_opened"))

print("B. Late rate with vs without intervention")
display(summary(valid.assign(intervention=valid["intervention_count"] > 0), "intervention"))
# Fairer comparison: among orders that were ALREADY in trouble at pickup
at_risk = valid[valid["pickup_overrun_min"] > 15]
print(f"   ...among at-risk orders only (pickup overrun > 15 min, n={len(at_risk)}):")
display(summary(at_risk.assign(operational=at_risk["operational_intervention"]), "operational"))

print("C. Late rate by intervention type (and whether it came before pickup)")
typ = valid.merge(ivt[["order_id", "intervention_type", "before_pickup"]], on="order_id")
display(summary(typ, ["intervention_type", "before_pickup"]))

print("D. Restaurants contributing the most late orders")
rest = (valid.groupby("restaurant_id")
             .agg(orders=("order_id", "count"), late_orders=("late_10", "sum"), late_rate=("late_10", "mean"),
                  median_pickup_overrun=("pickup_overrun_min", "median"))
             .sort_values("late_orders", ascending=False))
rest["share_of_all_late"] = rest["late_orders"] / rest["late_orders"].sum()
rest["cumulative_share"] = rest["share_of_all_late"].cumsum()
display(rest.head(10).round(3))
print(f"   Top 10 restaurants = {rest['share_of_all_late'].head(10).sum():.0%} of late orders "
      f"({10/len(rest):.0%} of restaurants)")

print("E. Journeys with support + intervention + still late")
e = valid[valid["support_opened"] & (valid["intervention_count"] > 0) & valid["late_10"]]
print(f"   {len(e)} orders")
display(e[["order_id", "intervention_types", "delay_min", "pickup_overrun_min"]]
        .sort_values("delay_min", ascending=False).head(10))
display(e["intervention_types"].value_counts().to_frame("orders"))

A. Support interaction vs delay


,orders,late_rate,median_delay
support_opened,,,
False,1203,0.135,-0.2
True,292,0.637,12.3


B. Late rate with vs without intervention


,orders,late_rate,median_delay
intervention,,,
False,1091,0.234,1.4
True,404,0.233,1.5


   ...among at-risk orders only (pickup overrun > 15 min, n=385):


...

D. Restaurants contributing the most late orders


,orders,late_orders,late_rate,median_pickup_overrun,share_of_all_late,cumulative_share
restaurant_id,,,,,,
R004,30,11,0.367,11.989,0.032,0.032
R020,27,11,0.407,10.647,0.032,0.063
R045,29,11,0.379,12.816,0.032,0.095
R018,27,11,0.407,11.233,0.032,0.126
R024,25,10,0.400,11.767,0.029,0.155
R023,32,10,0.312,10.053,0.029,0.184
R050,35,10,0.286,10.193,0.029,0.213
R041,32,9,0.281,8.882,0.026,0.239
R040,28,9,0.321,7.902,0.026,0.264


   Top 10 restaurants = 29% of late orders (17% of restaurants)
E. Journeys with support + intervention + still late
   46 orders


,order_id,intervention_types,delay_min,pickup_overrun_min
100,O00101,CUSTOMER_CREDIT,85.51,8.066400
197,O00198,RESTAURANT_CONTACT,35.23,37.969243
496,O00497,PRIORITY_DISPATCH,31.78,34.893574
612,O00613,PRIORITY_DISPATCH,27.18,32.212297
7,O00008,CUSTOMER_CREDIT,26.86,31.008935
301,O00302,PRIORITY_DISPATCH,26.32,32.246389
543,O00544,DRIVER_REASSIGNMENT,24.92,27.502537
861,O00862,DRIVER_REASSIGNMENT,24.10,32.527033
538,O00539,DRIVER_REASSIGNMENT,23.28,29.643518
1577,O01578,PRIORITY_DISPATCH,22.76,32.657117


,orders
intervention_types,
DRIVER_REASSIGNMENT,17
CUSTOMER_CREDIT,10
PRIORITY_DISPATCH,10
RESTAURANT_CONTACT,9


### My answer

- **A.** Support orders: 64% late vs 14%. *Tells:* support is a good warning sign. *Doesn't prove:* support causes delay (probably the other way around).
- **B.** With vs without intervention: 23.3% vs 23.4%, no difference. *Doesn't prove* interventions don't work, because they are used on the worst orders.
- **C.** Before pickup, every intervention type is 28 to 40% late. After-pickup ones look great, but only because those orders were already picked up fast.
- **D.** Top 10 restaurants (17%) = 29% of late orders (R004, R020, R045, R018). *Doesn't prove* it's the restaurant's fault; it could be the driver arriving late.
- **E.** 46 orders had support + intervention + still late. Most had pickup 27 to 38 min over, so the intervention came too late.

# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

### My answer

**KPI:** late rate 23.3% → **Outcome:** M1 → **Workflow:** M2 pickup overrun, M3 customer contact → **Interventions:** M4, M5 → **Data:** orders, dispatch, driver events, app actions, tickets, interventions, outcomes

1. Controllable by ops: M4 (respond to every frustrated customer), M5 (act before pickup), partly M2
2. Outcomes: M1, M3
3. Missing event that limits me most: **driver arrived at restaurant** (and food ready). Without it I can't say if the pickup delay is the restaurant or the driver.
4. Instrument next: arrival + food ready events, an intervention log that matches dispatch, and a test group to measure if interventions work.

**Limitation:** interventions aren't random and the log disagrees with dispatch, so the model shows *where* delay happens but not *whether interventions work*.

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.